# Predicción sobre test (submission 05)

Aplica a `data/test_data.jsonlines` el modelo que guardó `01_entrenamiento.ipynb` en `entrega/modelo/` y genera `entrega/submission.csv`, idéntico al archivo enviado a Kaggle (`05_vendedor_tiempo_frases.csv`).

**Usa `features.py`** para las features fila por fila. No ajusta ni recalcula nada: los codificadores y el modelo se cargan ya ajustados con train.

## 1. Librerías y rutas

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from features import construir_features

TEST = Path("../data/test_data.jsonlines")
MODELO = Path("modelo")
SUBMISSION = Path("submission.csv")

## 2. Cargar lo que guardó el entrenamiento

In [2]:
codificador = joblib.load(MODELO / "codificador.joblib")
modelo = joblib.load(MODELO / "lightgbm.joblib")
columnas = json.loads((MODELO / "columnas.json").read_text(encoding="utf-8"))

## 3. Leer test

Sin reordenar ni filtrar: el `ID` de la submission es el número de fila.

In [3]:
test = pd.read_json(TEST, lines=True)
test.shape

(30000, 44)

## 4. Aplicar las transformaciones ajustadas con train

In [4]:
X_test = codificador.transform(construir_features(test))[columnas]
X_test.shape

(30000, 125)

## 5. Predecir y guardar la submission

Formato: `ID` = número de fila de test empezando en 1, `condition` = `new` o `used`.

In [5]:
submission = pd.DataFrame({"ID": np.arange(1, len(test) + 1), "condition": modelo.predict(X_test)})
submission.to_csv(SUBMISSION, index=False)
submission["condition"].value_counts()

condition
new     15881
used    14119
Name: count, dtype: int64

## 6. Verificar el formato

In [6]:
sub = pd.read_csv(SUBMISSION)
assert list(sub.columns) == ["ID", "condition"] and len(sub) == 30_000
assert (sub["ID"] == np.arange(1, 30_001)).all() and sub["condition"].isin(["new", "used"]).all()
print("Formato correcto: 30.000 filas, ID 1..30000, solo new/used")

Formato correcto: 30.000 filas, ID 1..30000, solo new/used
